<a href="https://colab.research.google.com/github/Croop-weed/DeepLearning-parctice-/blob/main/data_extractionv2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
!pip install pypdf langchain-community langchain-text-splitters pdfplumber langchain-openai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 52.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 102.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.9/121.9 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 54.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 558.5/558.5 kB 37.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 106.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 96.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 

In [5]:
import os
from uuid import uuid4
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document as LCDocument
import pdfplumber
from pypdf import PdfReader
import pandas as pd


/tmp/ipykernel_1309/2612690753.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [6]:
path = "/content/SmartBin_AI_Project_Spec.pdf"

In [7]:
def extract_pdf_metadata(pdf_path):

    with open(pdf_path, 'rb') as f:
        reader = PdfReader(f)
        info = reader.metadata
        page_count = len(reader.pages)

    return {
        "title": info.title if info.title else os.path.basename(pdf_path),
        "author": info.author if info.author else "Unknown",
        "creator": info.creator if info.creator else "Unknown",
        "page_count": page_count
    }

In [8]:
print(extract_pdf_metadata(path))

{'title': 'SmartBin_AI_Project_Spec', 'author': 'Unknown', 'creator': 'Unknown', 'page_count': 10}


In [9]:
def extract_content_and_tables(pdf_path):
    structured_pages = []
    full_text_list = []

    with pdfplumber.open(pdf_path) as pdf:
        for page_num, page in enumerate(pdf.pages, start=1):
            tables = page.extract_tables()
            text = page.extract_text(layout=False)

            page_text = text if text else ""
            full_text_list.append(page_text)

            structured_pages.append({
                "page_number": page_num,
                "text": page_text,
                "tables": tables if tables else []
            })

    entire_document_text = "\n--- PAGE BREAK ---\n".join(full_text_list)
    return entire_document_text, structured_pages

In [10]:
def decision_mind_pdf_loader(pdf_path, chunk_size=1000, chunk_overlap=150):

    metadata = extract_pdf_metadata(pdf_path)
    full_text, structured_pages = extract_content_and_tables(pdf_path)

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", " ", ""]
    )

    parent_doc = LCDocument(
        page_content=full_text,
        metadata={
            "source": os.path.basename(pdf_path),
            "title": metadata["title"]
        }
    )

    langchain_chunks = text_splitter.split_documents([parent_doc])

    return {
        "metadata": metadata,
        "full_text": full_text,
        "structured_pages": structured_pages,
        "vector_chunks": langchain_chunks
    }

In [12]:
!pip install -q langchain-google-genai pydantic


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.2/72.2 kB 1.4 MB/s eta 0:00:00


In [34]:
import os
from uuid import uuid4
from pydantic import BaseModel, Field
from langchain_google_genai import ChatGoogleGenerativeAI
from google.colab import userdata

# -------------------------------------------------------------------
# 1. API KEY SETUP
# -------------------------------------------------------------------
if "GOOGLE_API_KEY" not in os.environ:
    os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

# -------------------------------------------------------------------
# 2. PYDANTIC SCHEMA (Focuses ONLY on DecisionAnalysis fields)
# -------------------------------------------------------------------
class DecisionAnalysisSchema(BaseModel):
    summary: str = Field(description="A 2-3 sentence executive summary of the decision context.")
    pros: list[str] = Field(description="List of specific advantages or positive outcomes.")
    cons: list[str] = Field(description="List of specific disadvantages, costs, or trade-offs.")
    risks: list[str] = Field(description="List of potential future risks or vulnerabilities.")
    alternatives: list[str] = Field(description="List of alternative options or vendors considered.")
    assumptions: list[str] = Field(description="List of core assumptions or prerequisites.")
    confidence: float = Field(description="Confidence score between 0.0 and 1.0 on document clarity.")

# -------------------------------------------------------------------
# 3. PROMPT TEMPLATES
# -------------------------------------------------------------------
PROMPT_1_SUMMARY = """You are the Summarization Engine for DecisionMind.
Process the raw document text and create a structured summary aligned with the decision context provided by the user.

DECISION CONTEXT:
- Title: {title}
- Topic: {topic}
- Department: {department}
- Problem Statement: {problem_statement}
- Stated Decision: {decision}
- Stated Reason: {reason}

RAW DOCUMENT TEXT:
<raw_document_text>
{raw_pdf_text}
</raw_document_text>

Output a clean, dense Markdown summary covering Context, Key Findings, Resolution Alignment, and Noted Constraints.
"""

PROMPT_2_ANALYSIS = """You are the Strategic Reasoning Engine for DecisionMind.
Parse the document summary alongside decision record context to extract structured analysis details.

DECISION RECORD:
- Title: {title}
- Topic: {topic}
- Department: {department}
- Problem Statement: {problem_statement}
- Decision Taken: {decision}
- Reason: {reason}

DOCUMENT SUMMARY:
<document_summary>
{document_summary}
</document_summary>

Strictly extract all required fields according to the schema parameters.
"""

# -------------------------------------------------------------------
# 4. UNIFIED DECISIONMIND PIPELINE EXECUTION
# -------------------------------------------------------------------
def process_decisionmind_pipeline(
    pdf_path: str,
    user_decision_input: dict,  # Manually supplied by user via UI form
    mock_user_id: uuid4,
    prompt_version: str = "v1.0"
):
    # A. Run PDF extraction and chunking
    pipeline_output = decision_mind_pdf_loader(pdf_path)
    raw_pdf_text = pipeline_output["full_text"]

    # B. Initialize Gemini LLM instances
    gemini_text = ChatGoogleGenerativeAI(model="gemini-3.5-flash", temperature=0) # Changed model to gemini-1.0
    gemini_structured = gemini_text.with_structured_output(DecisionAnalysisSchema)

    # C. STAGE 1: Generate Document Summary using Gemini
    print("⏳ Stage 1: Summarizing PDF text with Gemini...")
    summary_prompt_formatted = PROMPT_1_SUMMARY.format(
        title=user_decision_input["title"],
        topic=user_decision_input["topic"],
        department=user_decision_input["department"],
        problem_statement=user_decision_input["problem_statement"],
        decision=user_decision_input["decision"],
        reason=user_decision_input["reason"],
        raw_pdf_text=raw_pdf_text
    )
    generated_summary = gemini_text.invoke(summary_prompt_formatted).content

    # D. STAGE 2: Generate DecisionAnalysis Model using Gemini
    print("🧠 Stage 2: Extracting Decision Analysis model with Gemini...")
    analysis_prompt_formatted = PROMPT_2_ANALYSIS.format(
        title=user_decision_input["title"],
        topic=user_decision_input["topic"],
        department=user_decision_input["department"],
        problem_statement=user_decision_input["problem_statement"],
        decision=user_decision_input["decision"],
        reason=user_decision_input["reason"],
        document_summary=generated_summary
    )
    analysis_result: DecisionAnalysisSchema = gemini_structured.invoke(analysis_prompt_formatted)

    # E. Build SQLAlchemy 'Decision' Record Payload
    decision_db_id = uuid4()
    decision_db_record = {
        "id": decision_db_id,
        "created_by": mock_user_id,
        "title": user_decision_input["title"],
        "topic": user_decision_input["topic"],
        "department": user_decision_input["department"],
        "problem_statement": user_decision_input["problem_statement"],
        "decision": user_decision_input["decision"],
        "reason": user_decision_input["reason"],
        "status": "APPROVED"
    }

    # F. Build SQLAlchemy 'Document' Record Payload (extracted_text stores the SUMMARY)
    document_db_record = {
        "id": uuid4(),
        "decision_id": decision_db_id,
        "uploaded_by": mock_user_id,
        "filename": os.path.basename(pdf_path),
        "stored_filename": f"{uuid4()}.pdf",
        "mime_type": "application/pdf",
        "file_path": pdf_path,
        "file_size": os.path.getsize(pdf_path) if os.path.exists(pdf_path) else 0,
        "document_type": "PDF",
        "extracted_text": generated_summary  # ✅ Storing the generated summary
    }

    # G. Build SQLAlchemy 'DecisionAnalysis' Record Payload
    decision_analysis_db_record = {
        "id": uuid4(),
        "decision_id": decision_db_id,
        "summary": analysis_result.summary,
        "pros": analysis_result.pros,
        "cons": analysis_result.cons,
        "risks": analysis_result.risks,
        "alternatives": analysis_result.alternatives,
        "assumptions": analysis_result.assumptions,
        "confidence": analysis_result.confidence,
        "model_name": "gemini-1.5-flash", # Changed model to gemini-1.
        "prompt_version": prompt_version
    }

    return {
        "decision_db_record": decision_db_record,
        "document_db_record": document_db_record,
        "decision_analysis_db_record": decision_analysis_db_record,
        "vector_chunks": pipeline_output["vector_chunks"]
    }

In [35]:
# =====================================================================
# STRUCTURED INPUT PAYLOAD (Frontend UI / API Request Body)
# =====================================================================

user_decision_input = {
    # 1. Core Identification & Categorization
    "title": "SmartBin AI: Deep Learning Garbage Overflow & Spillage Detection System",
    "topic": "Computer Vision / Edge AI",
    "department": "Municipal Operations & Public Works",

    # 2. Problem Statement (Underlying pain point triggering this decision)
    "problem_statement": (
        "Municipal solid waste collection in Indian cities operates on fixed schedules regardless of actual bin fill levels. "
        "This leads to two major failures: bins overflow before scheduled pickup, and once full, citizens throw garbage "
        "around the bin boundary, causing severe spillage and public health hazards."
    ),

    # 3. Decision Taken (The approved architecture / action item)
    "decision": (
        "Deploy a two-model computer vision pipeline on edge hardware (Raspberry Pi / Jetson Nano). "
        "Use MobileNetV2 via transfer learning for inside-bin fill-level classification and YOLOv8n for "
        "outside-bin spillage object detection, linked to a weighted priority scoring engine (Spill: 0.45, Fill: 0.35, Time: 0.20) "
        "and a FastAPI backend."
    ),

    # 4. Reason / Justification (Why this path was selected)
    "reason": (
        "Occupancy estimation is an image classification problem requiring no spatial localization, whereas spillage severity "
        "requires bounding-box area calculations. Splitting the workload into two specialized models avoids over-engineering "
        "the fill-level task while preserving spatial accuracy for spillage. MobileNetV2 and YOLOv8n optimize depthwise separable "
        "convolutions and single-stage detection to fit tight edge compute/thermal budgets."
    )
}

In [37]:
extracted_intelligence = process_decisionmind_pipeline(path, user_decision_input, uuid4())

⏳ Stage 1: Summarizing PDF text with Gemini...
🧠 Stage 2: Extracting Decision Analysis model with Gemini...


In [16]:
for key,value in extracted_intelligence:
  print("=====================================================")
  print(key)
  print("------------------------------------------------")
  print(value)
  print("=====================================================")

title
------------------------------------------------
SmartBin AI: Deep Learning-Based Garbage Overflow & Spillage Detection System
topic
------------------------------------------------
Edge AI
department
------------------------------------------------
Engineering
problem_statement
------------------------------------------------
Municipal solid waste collection runs on fixed schedules regardless of actual fill levels. This causes bins to overflow before collection and leads people to dump trash around full bins, causing public health hazards and spillage.
decision
------------------------------------------------
Implement a two-model computer vision system running on edge hardware (Raspberry Pi / Jetson Nano). Use MobileNetV2 for inside-bin fill-level classification and YOLOv8n for outside spillage object detection, coupled with a weighted priority scoring engine and FastAPI backend.
reason
------------------------------------------------
Occupancy classification does not require s